# Lab 7 – Logistic Regression
**Dataset:** Medical Insurance Cost (`insurance.csv`)  
**Target:** `smoker` — Binary classification (yes = 1 / no = 0)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 1. Load and Explore Data

In [ ]:
df = pd.read_csv('insurance.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
print('Class balance:')
print(df['smoker'].value_counts())
print()
print(df['smoker'].value_counts(normalize=True).round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.boxplot(data=df, x='smoker', y='charges', palette={'yes':'tomato','no':'steelblue'}, ax=axes[0])
axes[0].set_title('Charges by Smoker Status')

sns.boxplot(data=df, x='smoker', y='bmi', palette={'yes':'tomato','no':'steelblue'}, ax=axes[1])
axes[1].set_title('BMI by Smoker Status')

sns.boxplot(data=df, x='smoker', y='age', palette={'yes':'tomato','no':'steelblue'}, ax=axes[2])
axes[2].set_title('Age by Smoker Status')

plt.tight_layout(); plt.show()

## 2. Feature Preparation

In [ ]:
df_model = df.copy()
df_model['sex_enc'] = (df_model['sex'] == 'male').astype(int)
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True)
df_model.drop(columns=['sex'], inplace=True)

X = df_model.drop(columns=['smoker'])
y = (df_model['smoker'] == 'yes').astype(int)

print('Features:', X.columns.tolist())
print('Target distribution:', y.value_counts().to_dict())

## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y)

print(f'Training size: {X_train.shape}')
print(f'Test size    : {X_test.shape}')

## 4. Train Logistic Regression Model

In [ ]:
logmodel = LogisticRegression(max_iter=1000, random_state=42)
logmodel.fit(X_train, y_train)
print('Model trained successfully.')

## 5. Evaluate the Model

In [ ]:
predictions = logmodel.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, predictions,
                            target_names=['Non-Smoker','Smoker']))

In [ ]:
cm   = confusion_matrix(y_test, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Non-Smoker','Smoker'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout(); plt.show()

In [ ]:
coeff = pd.DataFrame({'Feature': X.columns,
                      'Coefficient': logmodel.coef_[0]})
coeff = coeff.reindex(coeff['Coefficient'].abs().sort_values(ascending=True).index)

coeff.plot(kind='barh', x='Feature', y='Coefficient', legend=False,
           color=coeff['Coefficient'].apply(lambda v: 'tomato' if v>0 else 'steelblue'),
           edgecolor='black', figsize=(9, 5))
plt.axvline(0, color='black', lw=0.8)
plt.title('Logistic Regression Coefficients\n(positive = more likely to be a smoker)')
plt.tight_layout(); plt.show()

**Interpretation:**  
`charges` dominates the model — high charges are the strongest signal that someone is a smoker. `bmi` is the second most important feature. The model achieves high precision and recall for non-smokers (the majority class) and reasonable performance on smokers. The confusion matrix shows the model rarely confuses smokers and non-smokers, confirming that insurance charges are a near-perfect proxy for smoking status in this dataset.